# D1 — fixed-step networks chained across the magnet: the grid read out

Loads the tables written by `aggregate.py` and the figures written by `plot.py`. Computes nothing: every number here is copied from a record on disk, so the write-up can cite the CSV and the commit.

In [ ]:
import os, pandas as pd
from IPython.display import Image, display
R, F = 'results', 'figures'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
status = pd.read_csv(os.path.join(R, 'status.csv'))
print('chains done: %d / %d;  all legs converged in %d' % (status.done.sum(), len(status), (status.all_legs_converged == True).sum()))

## The table: median endpoint error at z1 against the RK6 truth, test split [µm]

In [ ]:
pd.read_csv(os.path.join(R, 'table_vs_rk6_endpoint_test_pos_med_um.csv')).set_index('N')

## The same against the particle's real first-SciFi state (material included) [µm]

In [ ]:
pd.read_csv(os.path.join(R, 'table_vs_real_scifi_state_test_pos_med_um.csv')).set_index('N')

## The exact scheme at the same (N, q): the ceiling [µm]

In [ ]:
p = os.path.join(R, 'table_exact_scheme_test_pos_med_um.csv')
pd.read_csv(p).set_index('N') if os.path.exists(p) else 'run ../D2_Comparators first'

## Figures

In [ ]:
for f in ('heatmap_vs_rk6.png', 'error_vs_q.png', 'error_vs_N.png', 'growth_along_crossing.png', 'own_step_vs_inherited.png', 'vs_real_scifi.png'):
    p = os.path.join(F, f)
    if os.path.exists(p):
        print(f); display(Image(p))

## Per leg: what each leg adds against what it inherits

In [ ]:
per_leg = pd.read_csv(os.path.join(R, 'per_leg.csv'))
per_leg.groupby(['N','q'])[['converged','restarts','wall_s']].agg({'converged':'mean','restarts':'sum','wall_s':'sum'}).rename(columns={'converged':'fraction_converged'})

## Per momentum band, the best chain

In [ ]:
bands = pd.read_csv(os.path.join(R, 'by_p_band.csv'))
tab = pd.read_csv(os.path.join(R, 'chain_table.csv'))
best = tab[(tab.split=='test') & (tab.comparator=='vs_rk6_endpoint')].nsmallest(1, 'pos_med_um').iloc[0]
print('best chain: N = %d, q = %d' % (best.N, best.q))
bands[(bands.N==best.N) & (bands.q==best.q) & (bands.split=='test')].pivot(index='p_band', columns='comparator', values='pos_med_um')